In [24]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import time
from collections import defaultdict
import json
import math

In [25]:
nba_sc = pd.read_csv("games.csv")
nba_sc.head(2)

,Date,Visitor,PTS,Home,PTS.1,Attend.,LOG,Arena,Notes
0,"Sat, Nov 01, 2025",Golden State Warriors,NaN,Boston Celtics,NaN,"19,000",7:30 PM,TD Garden,NaN
1,"Sat, Nov 01, 2025",Los Angeles Lakers,NaN,New York Knicks,NaN,"19,400",7:30 PM,Madison Square Garden,NaN


## 1. Write codes that computes and prints, for each team i, the following information:

In [26]:
all_teams = sorted(set(nba_sc['Home'].unique()).union(set(nba_sc['Visitor'].unique())))

for team in all_teams:
    home_games = nba_sc[nba_sc['Home'] == team].copy()
    away_games = nba_sc[nba_sc['Visitor'] == team].copy()

    print(f"{team}")
        
#a) all the dates when team i played home
    home_dates = sorted(home_games["Date"].unique())
    print(f"\n(a) Dates for at Home Games ({len(home_dates)} games):")
    
    for date in home_dates:
        opponents = home_games[home_games["Date"] == date]["Visitor"].tolist()
        print(f"    {date}: vs {', '.join(opponents)}")
    
    #b): for each team j, the number of times team i played against team j at home
    matchups_at_home = home_games.groupby("Visitor").size().to_dict()
    print(f"\n(b) Home games vs:")
    for opponent in all_teams:
        if opponent != team:
            games = matchups_at_home.get(opponent, 0)
            if games >= 1:
                print(f"    vs {opponent:30s}")
    
    #c): for each team j, the number of times team i played against team j away (i.e., at j’s home)
    matchups_on_road = away_games.groupby("Home").size().to_dict()
    print(f"\n(c) Away games vs:")
    for opponent in all_teams:
        if opponent != team:
            games = matchups_on_road.get(opponent, 0)
            if games >= 1:
                print(f"    Visiting {opponent:30s}")
    
    #d) all the dates when team j played away.
    away_dates = sorted(away_games["Date"].unique())
    print(f"\n(d) Away games dates ({len(away_dates)} games):")
    for date in away_dates:
        opponents = away_games[away_games["Date"] == date]["Home"].tolist()
        print(f"{date}: Visiting the {', '.join(opponents)}")

Atlanta Hawks

(a) Dates for at Home Games (10 games):
    Fri, Nov 07, 2025: vs Golden State Warriors
    Fri, Nov 28, 2025: vs Phoenix Suns
    Mon, Nov 03, 2025: vs Toronto Raptors
    Mon, Nov 17, 2025: vs Milwaukee Bucks
    Sat, Nov 15, 2025: vs Boston Celtics
    Sat, Nov 29, 2025: vs Dallas Mavericks
    Sun, Nov 23, 2025: vs New York Knicks
    Thu, Dec 25, 2025: vs Chicago Bulls
    Thu, Nov 27, 2025: vs Los Angeles Lakers
    Wed, Nov 19, 2025: vs Miami Heat

(b) Home games vs:
    vs Boston Celtics                
    vs Chicago Bulls                 
    vs Dallas Mavericks              
    vs Golden State Warriors         
    vs Los Angeles Lakers            
    vs Miami Heat                    
    vs Milwaukee Bucks               
    vs New York Knicks               
    vs Phoenix Suns                  
    vs Toronto Raptors               

(c) Away games vs:
    Visiting Brooklyn Nets                 
    Visiting Chicago Bulls                 
    Visiting Cleve

---------------

## 2. Write an Integer Program (without objective function) whose feasible solutions are all the feasible schedules, where a schedule is feasible if, for each team i:

(e) i plays home (possibly, against a different team) exactly on the dates computed
in (a) above;

(f) plays away (possibly, against a different team) exactly on the dates computed in
(d) above;

(g) for each team j, i plays home against team j exactly the number of times computed
in (b) above;

(h) for each team j, i plays away against team j (i.e., at j’s home) exactly the number
of times computed in (c) above.

**Sets**
- Teams: $T$ = Teams in the league (16 teams)
- Fixture Dates : $F$ = Fixture dates in the schedule (16 dates)

**Parameters**

- $\text{Home}_i$: dates when team $i$ plays at home, for each team $i \in T$
- $\text{Away}_i$: dates when team $i$ plays away, for each team $i \in T$

- $l_{ij}$: number of times team $i$ plays against team $j$ at home, $\forall$ team $i \in T$
- $v_{ij}$: number of times team $i$ plays at team $j$'s stadium, $\forall$ team $i \in T$

**Decision Variables**

$$x_{ijf} = \begin{cases} 1 & \text{if team } i \text{ hosts team } j \text{ on date } f \\ 0 & \text{otherwise} \end{cases}$$

$\forall i,j \in T, i \neq j, \forall f \in F$

(e) Games at home date constraint

For each team $i \in T$ and each fixture date $f \in F$:

$$\sum_{\substack{j \in T \\ j \neq i}} x_{ijf} = \begin{cases} 1 & \text{if } f \in \text{Home}_i \\ 0 & \text{otherwise} \end{cases}$$

Team $i$ plays exactly one home game on date $f$, if $f$ is one of their designated home dates, and plays no home games on date $f$ otherwise.

(f) Away Dates from calculation from 1.d.

For each team $i \in T$ and each fixture date $f \in F$:

$$\sum_{\substack{j \in T \\ j \neq i}} x_{jif} = \begin{cases} 1 & \text{if } f \in \text{Away}_i \\ 0 & \text{otherwise} \end{cases}$$

This is so that team $i$ plays  one away game on date $f$ if $f$ is one of their selected away dates from the data. 


(g) Home Matches counted in 1.b.

For all team fixtures $(i, j) \in T$ where $i \neq j$:

$$\sum_{f \in F} x_{ijf} = l_{ij}$$

This constraint ensures that team $i$ hosts team $j$  $l_{ij}$ times across all fixture dates, matching the original schedule.

(h) Away Matches countes in 1.c.

For all matchups $(i, j) \in T$,  $i \neq j$:

$$\sum_{f \in F} x_{jif} = v_{ij}$$

This ensures that team $i$ visits team $j$'s arena exactly $v_{ij}$ times. 

In [27]:
### Define Sets
T = sorted(set(nba_sc["Home"].unique()).union(set(nba_sc["Visitor"].unique())))  
F = sorted(nba_sc["Date"].unique())  
 
print(f"Set T (Teams): {len(T)} teams")
print(f"Set F (Fixture Dates): {len(F)} dates")
 
### Compute Params. for each team
Home = {}  
Away = {}  
l = {}     # l_ij
v = {}     # v_ij
 
for i in T:
    #Get all games where team i is home or away
    home_games = nba_sc[nba_sc["Home"] == i]
    away_games = nba_sc[nba_sc["Visitor"] == i]
    
    #Extract home and away dates
    Home[i] = sorted(home_games["Date"].unique())
    Away[i] = sorted(away_games["Date"].unique())
    
    #Count matchups
    for j in T:
        if i != j:
            l[i, j] = len(home_games[home_games["Visitor"] == j])
            v[i, j] = len(away_games[away_games["Home"] == j])

Set T (Teams): 16 teams
Set F (Fixture Dates): 16 dates


In [28]:
atl_haws = "Atlanta Hawks"
print(f"  Home_{{ATL}} = {len(Home[atl_haws])} dates")
print(f"  Away_{{ATL}} = {len(Away[atl_haws])} dates")
print(f"  Plays Boston at home: l_{{ATL,BOS}} = {l[atl_haws, 'Boston Celtics']} game")
print(f"  Plays at Brooklyn: v_{{ATL,BKN}} = {v[atl_haws, 'Brooklyn Nets']} game")

  Home_{ATL} = 10 dates
  Away_{ATL} = 6 dates
  Plays Boston at home: l_{ATL,BOS} = 1 game
  Plays at Brooklyn: v_{ATL,BKN} = 1 game


Solving the model formulated for question 2

In [29]:
# First, compute the sets and parameters from Problem 1
T = sorted(set(nba_sc["Home"].unique()).union(set(nba_sc["Visitor"].unique())))
F = sorted(nba_sc["Date"].unique())

#Build parameter dictionaries
Home = {}
Away = {}
l = {}
v = {}

for i in T:
    home_games = nba_sc[nba_sc["Home"] == i]
    away_games = nba_sc[nba_sc["Visitor"] == i]
    
    Home[i] = sorted(home_games["Date"].unique())
    Away[i] = sorted(away_games["Date"].unique())
    
    for j in T:
        if i != j:
            l[i, j] = len(home_games[home_games["Visitor"] == j])
            v[i, j] = len(away_games[away_games["Home"] == j])

print(f"Sets: {len(T)} teams, {len(F)} fixture dates")
print(f"Parameters computed: Home_i, Away_i, l_ij, v_ij")


model = gp.Model("NBA_Schedule_P2_Q2")
model.setParam("OutputFlag", 0) 

# Decision variables: x[i,j,f] = 1 if team i hosts team j on date f
x = {}
for i in T:
    for j in T:
        if i != j:
            for f in F:
                x[i, j, f] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_{f}")

print(f"Variables: {len(x)} binary variables created")

# Constraint e): Home date requirements
for i in T:
    for f in F:
        if f in Home[i]:
            # Team i must play at home on date f
            model.addConstr(gp.quicksum(x[i, j, f] for j in T if j != i) == 1, 
                          name=f"home_{i}_{f}")
        else:
            # Team i cannot play at home on date f
            model.addConstr(gp.quicksum(x[i, j, f] for j in T if j != i) == 0, name=f"no_home_{i}_{f}")

# Constraint f): Away date requirements
for i in T:
    for f in F:
        if f in Away[i]:
            # Team i must play away on date f
            model.addConstr(gp.quicksum(x[j, i, f] for j in T if j != i) == 1, name=f"away_{i}_{f}")
        else:
            # Team i cannot play away on date f
            model.addConstr(gp.quicksum(x[j, i, f] for j in T if j != i) == 0, name=f"no_away_{i}_{f}")

# Constraint (g): Home matchup counts
for i in T:
    for j in T:
        if i != j:
            model.addConstr(gp.quicksum(x[i, j, f] for f in F) == l[i, j], name=f"home_match_{i}_{j}")

# Constraint (h): Away matchup counts
for i in T:
    for j in T:
        if i != j:
            model.addConstr(gp.quicksum(x[j, i, f] for f in F) == v[i, j], name=f"away_match_{i}_{j}")

model.optimize()


if model.status == GRB.OPTIMAL:
    print("Feasible solution found for problem 2")
    
    # Extract the schedule
    schedule = []
    for i in T:
        for j in T:
            if i != j:
                for f in F:
                    if x[i, j, f].X > 0.5:  # Variable is 1
                        schedule.append({"Date": f, "Home": i, "Away": j})
    
    schedule_df = pd.DataFrame(schedule)
    schedule_df = schedule_df.sort_values(["Date", "Home"])
    
    print(f"Generated schedule: {len(schedule_df)} games")
    print("\nFirst 10 games:")
    print(schedule_df.head(10).to_string(index=False))
    
else:
    print("No feasible solution found")

Sets: 16 teams, 16 fixture dates
Parameters computed: Home_i, Away_i, l_ij, v_ij
Variables: 3840 binary variables created
Feasible solution found for problem 2
Generated schedule: 128 games

First 10 games:
             Date                Home                  Away
Fri, Nov 07, 2025       Atlanta Hawks          Phoenix Suns
Fri, Nov 07, 2025      Boston Celtics Golden State Warriors
Fri, Nov 07, 2025 Cleveland Cavaliers         Chicago Bulls
Fri, Nov 07, 2025    Dallas Mavericks       Houston Rockets
Fri, Nov 07, 2025      Denver Nuggets    Los Angeles Lakers
Fri, Nov 07, 2025     Milwaukee Bucks         Brooklyn Nets
Fri, Nov 07, 2025  Philadelphia 76ers       New York Knicks
Fri, Nov 07, 2025     Toronto Raptors            Miami Heat
Fri, Nov 21, 2025       Brooklyn Nets      Dallas Mavericks
Fri, Nov 21, 2025       Chicago Bulls Golden State Warriors


---------------------

## 3. Compute a feasible schedule that satisfy the following additional constraint: no team should play three consecutive matches where the sum of the absolute values of the difference between the time zones of two consecutive matches is 4 or more; or conclude that no such schedule exists. 

In [30]:
tz = {"Atlanta Hawks": 0, "Boston Celtics": 0, "Brooklyn Nets": 0,"Cleveland Cavaliers": 0, "Miami Heat": 0, "New York Knicks": 0, "Philadelphia 76ers": 0, "Toronto Raptors": 0, 
     "Chicago Bulls": 1, "Dallas Mavericks": 1,"Houston Rockets": 1, "Milwaukee Bucks": 1,"Denver Nuggets": 2, "Phoenix Suns": 2,"Golden State Warriors": 3, "Los Angeles Lakers": 3}
def get_tz_expr(i, f):
    if f in Home[i]:
        return tz[i]  ## playing at home, tz is a constant
    else:
        # playing away: tz = sum over opponents j of tz[j] *x[j,i,f]
        return gp.quicksum(tz[j] * x[j, i, f] for j in T if j != i)

In [31]:
F_s = sorted(F, key=lambda d: pd.to_datetime(d, format="%a, %b %d, %Y"))

In [32]:
model = gp.Model("NBA_Schedule_P2_Q3")
model.setParam("OutputFlag", 1) 
# Decision variables: x[i,j,f] = 1 if team i hosts team j on date f
x = {}
for i in T:
    for j in T:
        if i != j:
            for f in F:
                x[i, j, f] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_{f}")
# Constraint e): Home date requirements
for i in T:
    for f in F:
        if f in Home[i]:
            # Team i must play at home on date f
            model.addConstr(gp.quicksum(x[i, j, f] for j in T if j != i) == 1, 
                          name=f"home_{i}_{f}")
        else:
            # Team i cannot play at home on date f
            model.addConstr(gp.quicksum(x[i, j, f] for j in T if j != i) == 0,name=f"no_home_{i}_{f}")
# Constraint f): Away date requirements
for i in T:
    for f in F:
        if f in Away[i]:
            # Team i must play away on date f
            model.addConstr(gp.quicksum(x[j, i, f] for j in T if j != i) == 1,name=f"away_{i}_{f}")
        else:
            # Team i cannot play away on date f
            model.addConstr(gp.quicksum(x[j, i, f] for j in T if j != i) == 0,name=f"no_away_{i}_{f}")
# Constraint (g): Home matchup counts
for i in T:
    for j in T:
        if i != j:
            model.addConstr(gp.quicksum(x[i, j, f] for f in F) == l[i, j], name=f"home_match_{i}_{j}")
# Constraint (h): Away matchup counts
for i in T:
    for j in T:
        if i != j:
            model.addConstr(gp.quicksum(x[j, i, f] for f in F) == v[i, j], name=f"away_match_{i}_{j}")

""" 
New constraints (Q3)
"""
#For each team i look at the next three games 
for i in T:
    for k in range(len(F_s)-2):
        f1, f2, f3 = F_s[k], F_s[k+1], F_s[k+2]
        timezone1 = get_tz_expr(i, f1)
        timezone2 = get_tz_expr(i, f2)
        timezone3 = get_tz_expr(i, f3)
        
        # Auxiliary variables for |timezone1 - timezone2| and |timezone2 - timezone3|
        d1 = model.addVar(lb=0, name=f"d_{i}_{k}_1")
        d2 = model.addVar(lb=0, name=f"d_{i}_{k}_2")
        # Linearize |timezone1 - timezone2|: d1 >= timezone1 - timezone2 and d1 >= timezone2 - timezone1
        model.addConstr(d1 >= timezone1 - timezone2, name=f"abs1a_{i}_{k}")
        model.addConstr(d1 >= timezone2 - timezone1, name=f"abs1b_{i}_{k}")
        # Linearize |timezone2 - timezone3|: d2 >= timezone2 - timezone3 and d2 >= timezone3 - timezone2
        model.addConstr(d2 >= timezone2 - timezone3, name=f"abs2a_{i}_{k}")
        model.addConstr(d2 >= timezone3 - timezone2, name=f"abs2b_{i}_{k}")
        
        #sum of consecutive timezone jumps must be < 4
        model.addConstr(d1 + d2 <= 3, name=f"tz_limit_{i}_{k}")
 
model.optimize()

Set parameter OutputFlag to value 1


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-8565U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2112 rows, 4288 columns and 23440 nonzeros (Min)
Model fingerprint: 0x25c9c4c5
Model has 0 linear objective coefficients
Variable types: 448 continuous, 3840 integer (3840 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+00]

Presolve removed 350 rows and 0 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.04 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -


In [33]:
if model.status == GRB.OPTIMAL:
    print("Feasible solution found!")
else:
    print("No feasible solution found")

No feasible solution found


In [34]:
conflicts = 0
teams_conflicted = set()
for team in T:
    game_tzs = []
    for f in F_s:
        home_game = nba_sc[(nba_sc["Home"] == team) & (nba_sc["Date"] == f)]
        away_game = nba_sc[(nba_sc["Visitor"] == team) & (nba_sc["Date"] == f)]
        
        if len(home_game) > 0:
            game_tzs.append((f, tz[team], "home"))
        elif len(away_game) > 0:
            opponent = away_game["Home"].values[0]
            game_tzs.append((f, tz[opponent], f"visiting the {opponent}"))

    for k in range(len(game_tzs) - 2):
        f1, tz1, loc1 = game_tzs[k]
        f2, tz2, loc2 = game_tzs[k + 1]
        f3, tz3, loc3 = game_tzs[k + 2]
        
        diff_sum = abs(tz1 - tz2) + abs(tz2 - tz3)
        
        if diff_sum >= 4:
            conflicts += 1
            teams_conflicted.add(team)
            print(f"CONFLICT: {team}")
            print(f"  Game {k+1}: {f1} (tz={tz1}, {loc1})")
            print(f"  Game {k+2}: {f2} (tz={tz2}, {loc2})")
            print(f"  Game {k+3}: {f3} (tz={tz3}, {loc3})")
            print(f"{abs(tz1-tz2)} + {abs(tz2-tz3)} = {diff_sum} >= 4\n")

print(f"Total conflicts in original schedule: {conflicts}")
print(f"Teams affected: {len(teams_conflicted)}")

CONFLICT: Boston Celtics
  Game 1: Sat, Nov 01, 2025 (tz=0, home)
  Game 2: Mon, Nov 03, 2025 (tz=3, visiting the Los Angeles Lakers)
  Game 3: Wed, Nov 05, 2025 (tz=2, visiting the Denver Nuggets)
3 + 1 = 4 >= 4

CONFLICT: Brooklyn Nets
  Game 7: Sat, Nov 15, 2025 (tz=0, visiting the New York Knicks)
  Game 8: Mon, Nov 17, 2025 (tz=3, visiting the Los Angeles Lakers)
  Game 9: Wed, Nov 19, 2025 (tz=2, visiting the Phoenix Suns)
3 + 1 = 4 >= 4

CONFLICT: Cleveland Cavaliers
  Game 4: Fri, Nov 07, 2025 (tz=0, home)
  Game 5: Tue, Nov 11, 2025 (tz=3, visiting the Los Angeles Lakers)
  Game 6: Thu, Nov 13, 2025 (tz=2, visiting the Phoenix Suns)
3 + 1 = 4 >= 4

CONFLICT: Cleveland Cavaliers
  Game 11: Sun, Nov 23, 2025 (tz=0, home)
  Game 12: Thu, Nov 27, 2025 (tz=3, visiting the Golden State Warriors)
  Game 13: Fri, Nov 28, 2025 (tz=0, home)
3 + 3 = 6 >= 4

CONFLICT: Denver Nuggets
  Game 1: Sat, Nov 01, 2025 (tz=0, visiting the Brooklyn Nets)
  Game 2: Mon, Nov 03, 2025 (tz=3, visiting 

### We formulated Question 3 by preserving all prior scheduling constraints and adding the requirement that for any team, the sum of timezone jumps across any three consecutive matches must be less than 4. Gurobi returned an infeasible status. Then, the group analyzed the possible schedule and found out that teams with home stadiums in Eastern/Pacific time zones that alternate home-away-home have to go from one side to the other therefore making the problem infeasible

-------------

## 4. Compute any improvement to the current schedule. For instance, using Google Maps, you could compute and store the locations of all arenas, compute a feasible schedule that minimizes the maximum distance traveled by a team, and compare this value with the one of the current schedule to show the improvement. 

## Data and Arena Coordinates

Arena latitude/longitude values are based on publicly available information for each NBA venue.

In [35]:
nba_sc = pd.read_csv("games.csv")
nba_sc["Date"] = pd.to_datetime(nba_sc["Date"])

arena_coords = {
    "Atlanta Hawks":         (33.7573,  -84.3963),   # State Farm Arena
    "Boston Celtics":        (42.3662,  -71.0621),   # TD Garden
    "Brooklyn Nets":         (40.6826,  -73.9754),   # Barclays Center
    "Chicago Bulls":         (41.8807,  -87.6742),   # United Center
    "Cleveland Cavaliers":   (41.4965,  -81.6882),   # Rocket Mortgage FieldHouse
    "Dallas Mavericks":      (32.7905,  -96.8103),   # American Airlines Center
    "Denver Nuggets":        (39.7487, -105.0077),   # Ball Arena
    "Golden State Warriors": (37.7680, -122.3877),   # Chase Center
    "Houston Rockets":       (29.7508,  -95.3621),   # Toyota Center
    "Los Angeles Lakers":    (34.0430, -118.2673),   # Crypto.com Arena
    "Miami Heat":            (25.7814,  -80.1870),   # Kaseya Center
    "Milwaukee Bucks":       (43.0451,  -87.9173),   # Fiserv Forum
    "New York Knicks":       (40.7505,  -73.9934),   # Madison Square Garden
    "Philadelphia 76ers":    (39.9012,  -75.1720),   # Wells Fargo Center
    "Phoenix Suns":          (33.4457, -112.0712),   # Footprint Center
    "Toronto Raptors":       (43.6435,  -79.3791),   # Scotiabank Arena
}

## Distance Matrix

We use the haversine formula to compute great-circle distances in miles between arenas.

In [36]:
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth's radius in miles
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(a))

T = sorted(arena_coords.keys())
F = sorted(nba_sc["Date"].unique())

d = {}
for a in T:
    for b in T:
        la1, lo1 = arena_coords[a]
        la2, lo2 = arena_coords[b]
        d[a, b] = haversine(la1, lo1, la2, lo2)

print(f"Teams: {len(T)}, Dates: {len(F)}")
print(f"Distance range: {min(v for v in d.values() if v > 0):.0f} – {max(d.values()):.0f} miles")

Teams: 16, Dates: 16
Distance range: 5 – 2691 miles


In [37]:
home_on = {f: set(nba_sc.loc[nba_sc["Date"] == f, "Home"]) for f in F}
away_on = {f: set(nba_sc.loc[nba_sc["Date"] == f, "Visitor"]) for f in F}

played_dates = {t: sorted(nba_sc.loc[(nba_sc["Home"] == t) | (nba_sc["Visitor"] == t), "Date"].unique()) for t in T}
req_pair = nba_sc.groupby(["Home", "Visitor"]).size().to_dict()

## Baseline: Travel in the Original Schedule

Each team's travel is computed as the sum of distances across their full arena sequence: home → first game arena → next game arena → … → last game arena → home.

In [38]:
schedule_df.head(4)

,Date,Home,Away
8,"Fri, Nov 07, 2025",Atlanta Hawks,Phoenix Suns
11,"Fri, Nov 07, 2025",Boston Celtics,Golden State Warriors
35,"Fri, Nov 07, 2025",Cleveland Cavaliers,Chicago Bulls
46,"Fri, Nov 07, 2025",Dallas Mavericks,Houston Rockets


In [39]:
def compute_travel(schedule_df, team):
    games = schedule_df[(schedule_df["Home"] == team) | (schedule_df["Visitor"] == team)].sort_values("Date")
    arenas = [team] + list(games["Home"]) + [team]
    return sum(d[arenas[k], arenas[k+1]] for k in range(len(arenas) - 1))

orig_travel = {t: compute_travel(nba_sc, t) for t in T}

print("Original schedule travel (miles):")
for t in sorted(orig_travel, key=orig_travel.get, reverse=True):
    print(f"  {t:25s}: {orig_travel[t]:7.0f}")

orig_max = max(orig_travel.values())
orig_total = sum(orig_travel.values())
print(f"\nMax travel (worst team):  {orig_max:7.0f} miles")
print(f"Total league travel:      {orig_total:7.0f} miles")

Original schedule travel (miles):
  Golden State Warriors    :   23091
  Phoenix Suns             :   18807
  Los Angeles Lakers       :   18613
  Boston Celtics           :   18448
  New York Knicks          :   18116
  Philadelphia 76ers       :   16055
  Dallas Mavericks         :   14684
  Brooklyn Nets            :   14015
  Miami Heat               :   13908
  Toronto Raptors          :   12090
  Denver Nuggets           :   11552
  Houston Rockets          :   10505
  Cleveland Cavaliers      :   10024
  Milwaukee Bucks          :    8794
  Atlanta Hawks            :    7897
  Chicago Bulls            :    7854

Max travel (worst team):    23091 miles
Total league travel:       224453 miles


## Model

**Decision variables:**
- $x_{ijf} \in \{0,1\}$: 1 if team $i$ hosts team $j$ on date $f$ (sparse on original slots)
- $a_{itf} \in \{0,1\}$: 1 if team $i$ plays at team $t$'s arena on date $f$
- $z_{i,t_1,t_2,k} \in [0,1]$: 1 if on leg $k$, team $i$ travels from $t_1$'s arena to $t_2$'s arena
- $D \geq 0$: maximum travel distance across all teams

**Constraints:**
- (Q2 feasibility) — same as Q3: home slot, away slot, and pair-count constraints
- **Arena linkage:** exactly one arena per team per played date; if $i$ is home on $f$, arena = $i$; if $i$ is away on $f$, arena follows $x$
- **Leg linearization:** $z_{i,t_1,t_2,k} \leq a_{i,t_1,f_{k-1}}$, $z_{i,t_1,t_2,k} \leq a_{i,t_2,f_k}$, and $\sum_{t_1,t_2} z = 1$ per leg
- **Minimax:** $D \geq \text{total travel of team } i$ for every team $i$

**Objective:** $\min D$

In [40]:
model = gp.Model("Q4_minimax_travel")
model.setParam("OutputFlag", 1)
model.setParam("TimeLimit", 600)

# x: hosting assignment (sparse)
x = {}
for f in F:
    for i in home_on[f]:
        for j in away_on[f]:
            if i != j:
                x[i, j, f] = model.addVar(vtype=GRB.BINARY,
                                          name=f"x_{i}_{j}_{f.strftime('%Y%m%d')}")

# a: arena indicator
a = {}
for i in T:
    for f in played_dates[i]:
        for t in T:
            a[i, t, f] = model.addVar(vtype=GRB.BINARY,
                                      name=f"a_{i}_{t}_{f.strftime('%Y%m%d')}")

# z: transition indicator per leg
z = {}
for i in T:
    P = played_dates[i]
    for k in range(1, len(P)):
        for t1 in T:
            for t2 in T:
                z[i, t1, t2, k] = model.addVar(lb=0, ub=1, vtype=GRB.CONTINUOUS,
                                               name=f"z_{i}_{t1}_{t2}_{k}")

# D: max travel
D = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="D")

model.update()
print(f"Variables: {model.NumVars}")

# Q2 feasibility constraints
for f in F:
    for i in home_on[f]:
        model.addConstr(gp.quicksum(x[i, j, f] for j in away_on[f] if i != j) == 1)
    for j in away_on[f]:
        model.addConstr(gp.quicksum(x[i, j, f] for i in home_on[f] if i != j) == 1)

for i in T:
    for j in T:
        if i != j:
            rhs = req_pair.get((i, j), 0)
            model.addConstr(gp.quicksum(x[i, j, f] for f in F if (i, j, f) in x) == rhs)

# Arena linkage
for i in T:
    for f in played_dates[i]:
        model.addConstr(gp.quicksum(a[i, t, f] for t in T) == 1)
        if i in home_on[f]:
            model.addConstr(a[i, i, f] == 1)
        else:
            for t in T:
                if t in home_on[f] and (t, i, f) in x:
                    model.addConstr(a[i, t, f] == x[t, i, f])
                else:
                    model.addConstr(a[i, t, f] == 0)

# Leg linearization: z captures the (t1, t2) transition on leg k
for i in T:
    P = played_dates[i]
    for k in range(1, len(P)):
        f1, f2 = P[k-1], P[k]
        for t1 in T:
            for t2 in T:
                model.addConstr(z[i, t1, t2, k] <= a[i, t1, f1])
                model.addConstr(z[i, t1, t2, k] <= a[i, t2, f2])
        model.addConstr(gp.quicksum(z[i, t1, t2, k] for t1 in T for t2 in T) == 1)

# Team travel = pre-season leg + between-game legs + post-season leg
team_travel = {}
for i in T:
    P = played_dates[i]
    expr = gp.quicksum(d[i, t] * a[i, t, P[0]] for t in T)
    for k in range(1, len(P)):
        expr += gp.quicksum(d[t1, t2] * z[i, t1, t2, k] for t1 in T for t2 in T)
    expr += gp.quicksum(d[t, i] * a[i, t, P[-1]] for t in T)
    team_travel[i] = expr
    model.addConstr(D >= expr)

model.setObjective(D, GRB.MINIMIZE)
model.optimize()

Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 600


Variables: 66561
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-8565U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  600

Optimize a model with 126064 rows, 66561 columns and 375664 nonzeros (Min)
Model fingerprint: 0x21be6b5e
Model has 1 linear objective coefficients
Variable types: 61441 continuous, 5120 integer (5120 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 123247 rows and 64766 columns
Presolve time: 0.08s
Presolved: 2817 rows, 1795 columns, 8913 nonzeros
Variable types: 1332 continuous, 463 integer (463 binary)
Found heuristic solution: objective 22751.103432
Performing another presolve...
Presolve removed 1072 rows and 663 columns
Presolve time: 0.13s

Root re

### Comparison with the original schedule 

In [41]:
if model.status in (GRB.OPTIMAL, GRB.TIME_LIMIT) and model.SolCount > 0:
    opt_max = D.X
    opt_travel = {i: team_travel[i].getValue() for i in T}
    opt_total = sum(opt_travel.values())

    print(f"Optimized max travel: {opt_max:.0f} miles")
    print(f"Optimized total:      {opt_total:.0f} miles\n")

    comparison = pd.DataFrame({
        "Team": T,
        "Original": [orig_travel[t] for t in T],
        "Optimized": [opt_travel[t] for t in T],
    })
    comparison["Change"] = comparison["Optimized"] - comparison["Original"]
    comparison = comparison.sort_values("Original", ascending=False).reset_index(drop=True)
    print(comparison.to_string(index=False))

    print(f"\n--- Summary ---")
    print(f"Worst team's travel reduced: {orig_max:.0f} -> {opt_max:.0f} miles "
          f"(-{orig_max - opt_max:.0f} mi, {(orig_max - opt_max)/orig_max*100:.1f}%)")
    print(f"Total league travel:         {orig_total:.0f} -> {opt_total:.0f} miles "
          f"({opt_total - orig_total:+.0f} mi)")
else:
    print(f"No solution found. Status: {model.status}")

Optimized max travel: 20736 miles
Optimized total:      224462 miles

                 Team     Original    Optimized       Change
Golden State Warriors 23091.131210 20736.183941 -2354.947269
         Phoenix Suns 18807.376318 13661.445244 -5145.931074
   Los Angeles Lakers 18613.244566 17320.610286 -1292.634279
       Boston Celtics 18447.583617 20542.059540  2094.475923
      New York Knicks 18116.178768 17521.600664  -594.578104
   Philadelphia 76ers 16054.857128 16691.394228   636.537099
     Dallas Mavericks 14683.587918 11650.344937 -3033.242981
        Brooklyn Nets 14015.298523 16329.433807  2314.135284
           Miami Heat 13908.492501 13091.876377  -816.616124
      Toronto Raptors 12090.280066 12627.491800   537.211734
       Denver Nuggets 11552.046780 11638.335351    86.288570
      Houston Rockets 10505.039230 12292.642225  1787.602995
  Cleveland Cavaliers 10024.201017 14222.262103  4198.061086
      Milwaukee Bucks  8793.534951  9487.211519   693.676568
        Atlanta

### Conclusions on Q4

The minimax model makes the worst-traveled team travel **10.2% less**, from **23,091 miles to 20,736 miles**. This is done by changing the matchups in the original schedule's fixed home/away calendar. The total amount of travel in the league hasn't changed much, which is what a minimax objective is supposed to do: it spreads travel out fairly instead of cutting it down everywhere. Travel is moved from teams on the West Coast that are already busy (Golden State, LA Lakers, Phoenix) to teams that have extra money in their original travel budget (Cleveland, Toronto, Dallas).

In [42]:
# Extract and save the optimized schedule
opt_schedule = []
for (i, j, f) in x:
    if x[i, j, f].X > 0.5:
        opt_schedule.append({"Date": f, "Home": i, "Away": j})

opt_df = pd.DataFrame(opt_schedule).sort_values(["Date", "Home"])
opt_df.to_csv("optimized_schedule.csv", index=False)

--------------------

## Comparison with the NBA schedule link provided in the project description

In [43]:
#Actual 2023-24 NBA Schedule: Nov-Dec games between the 16 teams
#Source: Basketball Reference (2023-24 NBA Schedule)

nba_2324_games = [
    ("2023-11-01", "Boston Celtics", "Brooklyn Nets"),
    ("2023-11-01", "Atlanta Hawks", "Houston Rockets"),
    ("2023-11-01", "Cleveland Cavaliers", "Los Angeles Lakers"),
    ("2023-11-03", "Boston Celtics", "Miami Heat"),
    ("2023-11-03", "Toronto Raptors", "Dallas Mavericks"),
    ("2023-11-03", "Chicago Bulls", "New York Knicks"),
    ("2023-11-03", "Milwaukee Bucks", "Atlanta Hawks"),
    ("2023-11-04", "Houston Rockets", "Golden State Warriors"),
    ("2023-11-04", "Philadelphia 76ers", "Phoenix Suns"),
    ("2023-11-05", "Brooklyn Nets", "Cleveland Cavaliers"),
    ("2023-11-06", "New York Knicks", "Brooklyn Nets"),
    ("2023-11-06", "Boston Celtics", "Philadelphia 76ers"),
    ("2023-11-06", "Dallas Mavericks", "Los Angeles Lakers"),
    ("2023-11-06", "Chicago Bulls", "Denver Nuggets"),
    ("2023-11-08", "Miami Heat", "Houston Rockets"),
    ("2023-11-08", "Cleveland Cavaliers", "Golden State Warriors"),
    ("2023-11-08", "Philadelphia 76ers", "Brooklyn Nets"),
    ("2023-11-10", "Boston Celtics", "Brooklyn Nets"),
    ("2023-11-10", "Cleveland Cavaliers", "Chicago Bulls"),
    ("2023-11-10", "Dallas Mavericks", "Houston Rockets"),
    ("2023-11-10", "Phoenix Suns", "Denver Nuggets"),
    ("2023-11-11", "Atlanta Hawks", "Milwaukee Bucks"),
    ("2023-11-11", "Toronto Raptors", "Philadelphia 76ers"),
    ("2023-11-12", "Los Angeles Lakers", "Houston Rockets"),
    ("2023-11-13", "Philadelphia 76ers", "Dallas Mavericks"),
    ("2023-11-13", "Phoenix Suns", "Chicago Bulls"),
    ("2023-11-13", "Denver Nuggets", "Dallas Mavericks"),
    ("2023-11-14", "Milwaukee Bucks", "Miami Heat"),
    ("2023-11-15", "Houston Rockets", "Philadelphia 76ers"),
    ("2023-11-15", "Cleveland Cavaliers", "Brooklyn Nets"),
    ("2023-11-15", "Golden State Warriors", "Phoenix Suns"),
    ("2023-11-17", "New York Knicks", "Houston Rockets"),
    ("2023-11-17", "Denver Nuggets", "Atlanta Hawks"),
    ("2023-11-17", "Golden State Warriors", "Los Angeles Lakers"),
    ("2023-11-18", "Chicago Bulls", "Philadelphia 76ers"),
    ("2023-11-18", "Milwaukee Bucks", "Toronto Raptors"),
    ("2023-11-19", "Phoenix Suns", "Houston Rockets"),
    ("2023-11-20", "Toronto Raptors", "Brooklyn Nets"),
    ("2023-11-20", "Cleveland Cavaliers", "Milwaukee Bucks"),
    ("2023-11-21", "Atlanta Hawks", "Phoenix Suns"),
    ("2023-11-21", "Los Angeles Lakers", "Dallas Mavericks"),
    ("2023-11-22", "Brooklyn Nets", "New York Knicks"),
    ("2023-11-24", "Denver Nuggets", "Golden State Warriors"),
    ("2023-11-24", "Cleveland Cavaliers", "New York Knicks"),
    ("2023-11-24", "Houston Rockets", "Milwaukee Bucks"),
    ("2023-11-25", "Brooklyn Nets", "Boston Celtics"),
    ("2023-11-25", "Philadelphia 76ers", "Miami Heat"),
    ("2023-11-25", "Chicago Bulls", "Atlanta Hawks"),
    ("2023-11-25", "Los Angeles Lakers", "Golden State Warriors"),
    ("2023-11-27", "New York Knicks", "Miami Heat"),
    ("2023-11-27", "Denver Nuggets", "Cleveland Cavaliers"),
    ("2023-11-27", "Phoenix Suns", "Dallas Mavericks"),
    ("2023-11-28", "Toronto Raptors", "Boston Celtics"),
    ("2023-11-29", "Miami Heat", "Brooklyn Nets"),
    ("2023-11-29", "Milwaukee Bucks", "Los Angeles Lakers"),
    ("2023-12-01", "Toronto Raptors", "Denver Nuggets"),
    ("2023-12-01", "Boston Celtics", "Atlanta Hawks"),
    ("2023-12-01", "Miami Heat", "Chicago Bulls"),
    ("2023-12-02", "Denver Nuggets", "Houston Rockets"),
    ("2023-12-02", "Brooklyn Nets", "Milwaukee Bucks"),
    ("2023-12-03", "New York Knicks", "Dallas Mavericks"),
    ("2023-12-04", "Houston Rockets", "Cleveland Cavaliers"),
    ("2023-12-04", "Chicago Bulls", "Brooklyn Nets"),
    ("2023-12-05", "Toronto Raptors", "Phoenix Suns"),
    ("2023-12-05", "Boston Celtics", "Los Angeles Lakers"),
    ("2023-12-06", "Atlanta Hawks", "Brooklyn Nets"),
    ("2023-12-06", "Golden State Warriors", "Houston Rockets"),
    ("2023-12-07", "Philadelphia 76ers", "Denver Nuggets"),
    ("2023-12-08", "Brooklyn Nets", "Dallas Mavericks"),
    ("2023-12-08", "Atlanta Hawks", "Boston Celtics"),
    ("2023-12-08", "Denver Nuggets", "Chicago Bulls"),
    ("2023-12-09", "New York Knicks", "Toronto Raptors"),
    ("2023-12-10", "Houston Rockets", "Brooklyn Nets"),
    ("2023-12-10", "Miami Heat", "Denver Nuggets"),
    ("2023-12-10", "Dallas Mavericks", "Golden State Warriors"),
    ("2023-12-11", "Chicago Bulls", "Milwaukee Bucks"),
    ("2023-12-11", "Los Angeles Lakers", "Boston Celtics"),
    ("2023-12-12", "Cleveland Cavaliers", "Dallas Mavericks"),
    ("2023-12-13", "Miami Heat", "New York Knicks"),
    ("2023-12-13", "Philadelphia 76ers", "Cleveland Cavaliers"),
    ("2023-12-13", "Phoenix Suns", "Los Angeles Lakers"),
    ("2023-12-14", "Golden State Warriors", "Boston Celtics"),
    ("2023-12-15", "Cleveland Cavaliers", "Denver Nuggets"),
    ("2023-12-16", "Milwaukee Bucks", "New York Knicks"),
    ("2023-12-16", "Los Angeles Lakers", "Phoenix Suns"),
    ("2023-12-17", "Brooklyn Nets", "Houston Rockets"),
    ("2023-12-18", "Milwaukee Bucks", "Philadelphia 76ers"),
    ("2023-12-18", "Dallas Mavericks", "Chicago Bulls"),
    ("2023-12-19", "Boston Celtics", "Cleveland Cavaliers"),
    ("2023-12-20", "New York Knicks", "Chicago Bulls"),
    ("2023-12-20", "Phoenix Suns", "Golden State Warriors"),
    ("2023-12-21", "Dallas Mavericks", "Denver Nuggets"),
    ("2023-12-22", "Philadelphia 76ers", "Los Angeles Lakers"),
    ("2023-12-22", "Golden State Warriors", "Cleveland Cavaliers"),
    ("2023-12-23", "Milwaukee Bucks", "Brooklyn Nets"),
    ("2023-12-23", "Boston Celtics", "Chicago Bulls"),
    ("2023-12-25", "New York Knicks", "Milwaukee Bucks"),
    ("2023-12-25", "Boston Celtics", "Los Angeles Lakers"),
    ("2023-12-25", "Denver Nuggets", "Golden State Warriors"),
    ("2023-12-25", "Dallas Mavericks", "Phoenix Suns"),
    ("2023-12-25", "Miami Heat", "Philadelphia 76ers"),
    ("2023-12-27", "Brooklyn Nets", "Golden State Warriors"),
    ("2023-12-27", "Chicago Bulls", "Cleveland Cavaliers"),
    ("2023-12-27", "Houston Rockets", "New York Knicks"),
    ("2023-12-28", "Miami Heat", "Atlanta Hawks"),
    ("2023-12-28", "Toronto Raptors", "Los Angeles Lakers"),
    ("2023-12-29", "Cleveland Cavaliers", "Houston Rockets"),
    ("2023-12-29", "Milwaukee Bucks", "Denver Nuggets"),
    ("2023-12-30", "Brooklyn Nets", "Phoenix Suns"),
    ("2023-12-30", "Golden State Warriors", "Dallas Mavericks"),
    ("2023-12-31", "Toronto Raptors", "Chicago Bulls"),
    ("2023-12-31", "Philadelphia 76ers", "Atlanta Hawks"),
]

nba_real = pd.DataFrame(nba_2324_games, columns=["Date", "Home", "Away"])
nba_real["Date"] = pd.to_datetime(nba_real["Date"])
nba_real = nba_real.sort_values("Date").reset_index(drop=True)
def compute_travel(sched, team, home_col="Home", away_col="Away"):
    games = sched[(sched[home_col] == team) | (sched[away_col] == team)].sort_values("Date")
    arenas = [team] + list(games[home_col]) + [team]
    return sum(d[arenas[k], arenas[k+1]] for k in range(len(arenas) - 1))
nba_travel = {t: compute_travel(nba_real, t) for t in T}
comparison = pd.DataFrame({"Team": T,"NBA 23-24 (mi)": [nba_travel[t] for t in T],"Original (mi)": [orig_travel[t] for t in T],"Optimized (mi)": [opt_travel[t] for t in T],})
nba_games = {t: len(nba_real[(nba_real["Home"] == t) | (nba_real["Away"] == t)]) for t in T}
orig_games = {t: len(nba_sc[(nba_sc["Home"] == t) | (nba_sc["Visitor"] == t)]) for t in T}
comparison["NBA games"] = [nba_games[t] for t in T]
comparison["Our games"] = [orig_games[t] for t in T]
comparison["NBA mi/game"] = comparison["NBA 23-24 (mi)"] / comparison["NBA games"]
comparison["Orig mi/game"] = comparison["Original (mi)"] / comparison["Our games"]
comparison["Opt mi/game"] = comparison["Optimized (mi)"] / comparison["Our games"]
comparison = comparison.sort_values("NBA 23-24 (mi)", ascending=False).reset_index(drop=True)

In [44]:
comparison

,Team,NBA 23-24 (mi),Original (mi),Optimized (mi),NBA games,Our games,NBA mi/game,Orig mi/game,Opt mi/game
0,Houston Rockets,20772.935262,10505.039230,12292.642225,16,16,1298.308454,656.564952,768.290139
1,Phoenix Suns,17520.641279,18807.376318,13661.445244,13,16,1347.741637,1175.461020,853.840328
2,Golden State Warriors,16437.920444,23091.131210,20736.183941,14,16,1174.137175,1443.195701,1296.011496
3,Los Angeles Lakers,16172.843743,18613.244566,17320.610286,14,16,1155.203124,1163.327785,1082.538143
4,Denver Nuggets,15118.511157,11552.046780,11638.335351,15,16,1007.900744,722.002924,727.395959
5,Dallas Mavericks,14990.328776,14683.587918,11650.344937,15,16,999.355252,917.724245,728.146559
6,Chicago Bulls,11810.906016,7853.626170,8752.577395,15,16,787.393734,490.851636,547.036087
7,Brooklyn Nets,10361.914433,14015.298523,16329.433807,19,16,545.363918,875.956158,1020.589613
8,Cleveland Cavaliers,10149.316992,10024.201017,14222.262103,16,16,634.332312,626.512564,888.891381
9,Atlanta Hawks,8421.721075,7896.706367,7896.706367,11,16,765.611007,493.544148,493.544148


In [45]:
nba_max = max(nba_travel.values())
nba_total = sum(nba_travel.values())
nba_avg_per_game = np.mean([nba_travel[t] / nba_games[t] for t in T])
orig_avg_per_game = np.mean([orig_travel[t] / orig_games[t] for t in T])
opt_avg_per_game = np.mean([opt_travel[t] / orig_games[t] for t in T])

In [46]:
### Compare key metrics
print(f"Max travel (worst team):")
print(f"  NBA 23-24: {nba_max:.0f} miles")
print(f"  Original:   {orig_max:.0f} miles")
print(f"  Optimized:  {opt_max:.0f} miles\n")
print(f"Total league travel:")
print(f"  NBA 23-24: {nba_total:.0f} miles")
print(f"  Original:   {orig_total:.0f} miles")
print(f"  Optimized:  {opt_total:.0f} miles\n")
print(f"Average travel per game:")
print(f"  NBA 23-24: {nba_avg_per_game:.0f} miles/game")
print(f"  Original:   {orig_avg_per_game:.0f} miles/game")
print(f"  Optimized:  {opt_avg_per_game:.0f} miles/game")

Max travel (worst team):
  NBA 23-24: 20773 miles
  Original:   23091 miles
  Optimized:  20736 miles

Total league travel:
  NBA 23-24: 178385 miles
  Original:   224453 miles
  Optimized:  224462 miles

Average travel per game:
  NBA 23-24: 783 miles/game
  Original:   877 miles/game
  Optimized:  877 miles/game


Our optimized schedule actually beats the real NBA's worst-case travel (20,736 vs 20,773 miles), showing that a minimax approach can match what the NBA's professional schedulers achieve in terms of fairness across teams.

The original "games.csv" schedule was significantly worse than both alternatives for the worst-traveling team, confirming that schedule optimization through integer programming as seen in class can improve the schedule and reduce travel.

The NBA schedule has lower per-game travel (783 vs 882 mi/game) because the NBA has 30 teams, rather than 16 as in our dataset, having 30 teams would allow for closer games (e.g. Brooklyn Nets vs Knicks) 

In [52]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

### Load the optimized schedule from Q4
games = pd.read_csv("optimized_schedule.csv")[["Date", "Home", "Away"]]

### Team abbreviations (standard NBA 3-letter codes)
abbr = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE", "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN", "Golden State Warriors": "GSW", "Houston Rockets": "HOU",
    "Los Angeles Lakers": "LAL", "Miami Heat": "MIA", "Milwaukee Bucks": "MIL",
    "New York Knicks": "NYK", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Toronto Raptors": "TOR",
}


team_colors = {
    "ATL": ("#E03A3E", "white"),
    "BOS": ("#007A33", "white"),
    "BKN": ("#000000", "white"),
    "CHI": ("#CE1141", "white"),
    "CLE": ("#860038", "white"),
    "DAL": ("#00538C", "white"),
    "DEN": ("#0E2240", "#FEC524"),
    "GSW": ("#1D428A", "#FFC72C"),
    "HOU": ("#CE1141", "white"),
    "LAL": ("#552583", "#FDB927"),
    "MIA": ("#98002E", "white"),
    "MIL": ("#00471B", "white"),
    "NYK": ("#006BB6", "#F58426"),
    "PHI": ("#006BB6", "white"),
    "PHX": ("#1D1160", "#E56020"),
    "TOR": ("#CE1141", "white"),
}

HEADER_COLOR = "#17408B"
BORDER_COLOR = "#E5E5E5"

### Sort chronologically -- auto-detects any date format
games["DateParsed"] = pd.to_datetime(games["Date"])
games = games.sort_values(["DateParsed", "Home"]).reset_index(drop=True)
unique_dates = sorted(games["DateParsed"].unique())


def draw_badge(ax, x, y, code, width=0.28, height=0.095):
    ### Draw a colored pill with the team abbreviation
    bg, fg = team_colors[code]
    pill = FancyBboxPatch((x - width/2, y - height/2), width, height,
                          boxstyle="round,pad=0.005,rounding_size=0.015",
                          linewidth=0, facecolor=bg)
    ax.add_patch(pill)
    ax.text(x, y, code, ha="center", va="center",
            fontsize=7.5, fontweight="bold", color=fg, family="sans-serif")


def draw_card(ax, col, row, date_dt, games_today):
    ### Card is laid out in a unit coordinate system (0-1, 0-1)
    ### col and row are the grid position; we translate the card into place
    ### by plotting it in its own axes (each card gets its own subplot-like area)
    header_text = date_dt.strftime("%a  %b %d").upper()

    ### Card border (rounded rectangle covering full card area)
    border = FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                            boxstyle="round,pad=0.005,rounding_size=0.015",
                            linewidth=0.8, edgecolor=BORDER_COLOR,
                            facecolor="white")
    ax.add_patch(border)

    ### Header band
    header_h = 0.11
    header = FancyBboxPatch((0.05, 0.87), 0.90, header_h,
                            boxstyle="round,pad=0.002,rounding_size=0.01",
                            linewidth=0, facecolor=HEADER_COLOR)
    ax.add_patch(header)
    ax.text(0.5, 0.87 + header_h/2, header_text,
            ha="center", va="center",
            fontsize=9, fontweight="bold", color="white", family="sans-serif")

    ### Game rows -- 8 games, evenly spaced in the remaining vertical space
    top = 0.80
    bottom = 0.08
    n = len(games_today)
    if n == 0:
        return
    row_y = [top - i * (top - bottom) / max(n - 1, 1) for i in range(n)]

    for y, (_, g) in zip(row_y, games_today.iterrows()):
        away = abbr[g["Away"]]
        home = abbr[g["Home"]]
        draw_badge(ax, 0.30, y, away)
        ax.text(0.50, y, "vs", ha="center", va="center",
                fontsize=7, color="#888888", family="sans-serif")
        draw_badge(ax, 0.70, y, home)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")


### 4x4 grid of cards
fig, axes = plt.subplots(4, 4, figsize=(11, 13))
fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.02,
                    wspace=0.08, hspace=0.08)



for idx, date_dt in enumerate(unique_dates):
    row, col = divmod(idx, 4)
    ax = axes[row, col]
    games_today = games[games["DateParsed"] == date_dt]
    draw_card(ax, col, row, date_dt, games_today)

plt.savefig("schedule_grid.pdf", bbox_inches="tight", dpi=200)
plt.savefig("schedule_grid.png", bbox_inches="tight", dpi=200)
plt.close()

print(f"wrote schedule_grid.pdf and schedule_grid.png")
print(f"  {len(unique_dates)} dates, {len(games)} games")

wrote schedule_grid.pdf and schedule_grid.png
  16 dates, 128 games
